In [ ]:
pip install torch torchvision transformers pandas scikit-learn pillow # Needed for the baseline, ignore if already installed

In [1]:
import os
from dataclasses import dataclass
from os.path import basename, join, exists

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

/home/omar/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup
)

# ---- DATA PATHS ----
TRAIN_CSV = "data/train_images.csv"
TEST_CSV  = "data/test_images_path.csv"

TRAIN_IMG_DIR = "data/train_images"
TEST_IMG_DIR  = "data/test_images"

IMAGE_COLUMN = "image_path"
LABEL_COLUMN = "label"

NUM_LABELS = 200
BATCH_SIZE = 16
EPOCHS = 5

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Paths OK.")

Paths OK.


In [3]:
class BirdsDataset(Dataset):
    def __init__(self, df, img_dir, processor, image_column, label_column=None, label2id=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor
        self.image_column = image_column
        self.label_column = label_column
        self.label2id = label2id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        filename = basename(row[self.image_column])
        img_path = join(self.img_dir, filename)

        if not exists(img_path):
            raise FileNotFoundError(img_path)

        image = Image.open(img_path).convert("RGB")
        enc = self.processor(image, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.label_column and self.label2id:
            item["labels"] = torch.tensor(self.label2id[int(row[self.label_column])])

        return item

In [4]:
def create_dataloaders(processor):
    df = pd.read_csv(TRAIN_CSV)
    display(df.head())

    labels = sorted(df[LABEL_COLUMN].unique())
    label2id = {lab: i for i, lab in enumerate(labels)}
    id2label = {i: lab for lab, i in label2id.items()}

    print("Classes:", len(label2id))

    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df[LABEL_COLUMN],
        random_state=42
    )

    train_ds = BirdsDataset(train_df, TRAIN_IMG_DIR, processor, IMAGE_COLUMN, LABEL_COLUMN, label2id)
    val_ds   = BirdsDataset(val_df,   TRAIN_IMG_DIR, processor, IMAGE_COLUMN, LABEL_COLUMN, label2id)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    return train_loader, val_loader, label2id, id2label

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
train_loader, val_loader, label2id, id2label = create_dataloaders(processor)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


,image_path,label
0,/train_images/1.jpg,1
1,/train_images/2.jpg,1
2,/train_images/3.jpg,1
3,/train_images/4.jpg,1
4,/train_images/5.jpg,1


Classes: 200


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(label2id),
    id2label={i: str(id2label[i]) for i in id2label},
    label2id={str(k): v for k, v in label2id.items()},
    ignore_mismatched_sizes=True
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
num_training_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using: cpu
Training steps: 985


In [6]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * batch["labels"].size(0)
        preds = outputs.logits.argmax(-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            total_loss += loss.item() * batch["labels"].size(0)
            preds = outputs.logits.argmax(-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    return total_loss / total, correct / total

In [7]:
print("Device:", device)
print("Number of train batches:", len(train_loader))
print("Number of val batches:", len(val_loader))

Device: cpu
Number of train batches: 197
Number of val batches: 50


In [9]:
best_val = 0
best_path = f"{OUTPUT_DIR}/vit_baseline_best.pth"

for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train: loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"Val: loss={val_loss:.4f} acc={val_acc:.4f}")

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"Saved new best model (val_acc={best_val:.4f})")

print("Best val accuracy:", best_val)

KeyboardInterrupt: 

In [ ]:
best_model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(label2id),
    id2label={i: str(id2label[i]) for i in id2label},
    label2id={str(k): v for k, v in label2id.items()},
    ignore_mismatched_sizes=True
)
best_model.load_state_dict(torch.load(best_path, map_location=device))
best_model.to(device)
best_model.eval()

print("Loaded best model:", best_path)

In [ ]:
df_test = pd.read_csv(TEST_CSV)

test_ds = BirdsDataset(df_test, TEST_IMG_DIR, processor, IMAGE_COLUMN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = best_model(**batch).logits
        preds = logits.argmax(-1).cpu().tolist()
        all_preds.extend(preds)

mapped_preds = [id2label[i] for i in all_preds]
len(mapped_preds)

In [ ]:
submission = pd.DataFrame({
    "id": df_test["id"],
    "label": mapped_preds
})

submission.to_csv("baseline.csv", index=False)